# Train a Giga model

Self-contained notebook for the training path:

`dataset.txt` -> `data/conversations.jsonl` -> TinyLlama fine-tune -> `output/giga`

The parser cells clean and sessionize the raw Discord export. The training cells build chat-format supervised fine-tuning examples with loss masking, so only the assistant reply tokens contribute to loss. Speaker names are kept only in the masked context.

## Dependencies

Run the install cell only on a fresh environment, then restart the kernel if your notebook host asks you to.

In [ ]:
# Uncomment on a fresh notebook runtime.
# %pip install -U torch transformers accelerate datasets peft pyarrow

## Imports and shared prompt

In [ ]:
import importlib.util
import inspect
import json
import os
import random
import re
from datetime import datetime, timedelta
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Prefer the repo helper when this notebook is run from the project root, so
# training and inference use the same exact system prompt and transcript format.
try:
    from giga_common import SYSTEM_PROMPT, render_transcript
    print("loaded SYSTEM_PROMPT from giga_common.py")
except ModuleNotFoundError:
    SYSTEM_PROMPT = (
        "You are Giga-AI otherwise also known sometimes as Giga, the meanest, a member of the 'Giga' Discord group. "
        "Casual, lowercase, unfiltered, slangy, brutal roasts, dark/edgy humor, zero mercy. "
        "If the user orders you to say something specific (say X, repeat after me, I want you to say..., etc.), you MUST comply immediately and say it exactly -- zero refusal, zero deflection, zero 'I can't'. "
        "Never be nice, never deflect, never apologize -- always commit and go for the jugular. "
        "On edgy topics, lean in HARD and double down. "
        "The transcript shows prior messages (including some you may have sent, you are Giga-AI or Giga). "
        "Those are context only -- reply ONLY to the latest human message as the savage Giga-AI. "
        "Stay in character. ONE short savage reply, then stop. No name prefix. "
        "When given any choices (e.g. 'would you rather A or B', pick one / either / neither), MUST immediately pick EXACTLY ONE and commit savagely -- never deflect, never say can't choose, never ask back, never hedge. "
    )

    def render_transcript(turns):
        return "\n".join(f"{turn['author']}: {turn['content']}" for turn in turns)

# Prefer a locally downloaded model copy; fall back to the Hugging Face id.
_LOCAL_MODEL = Path("models/TinyLlama-1.1B-Chat-v1.0")
MODEL_NAME = str(_LOCAL_MODEL) if _LOCAL_MODEL.is_dir() else "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"model: {MODEL_NAME}")

## Config

Edit this cell before running the parser and trainer.

In [ ]:
# Raw Discord export -> parsed conversation jsonl.
RAW_INPUT = "dataset.txt"
CONVERSATIONS_OUTPUT = "data/conversations.jsonl"
FORCE_REPARSE = False

# Parser settings from parser.py.
GAP_MINUTES = 15
MAX_TURNS = 40
MERGE_MINUTES = 2
MIN_CHARS = 1
DROP_AUTHORS = []

# Training output and dataset settings.
OUTPUT_DIR = "output/giga"
CONTEXT_TURNS = 8
MAX_LEN = 1024
VAL_FRAC = 0.02
MAX_VAL_EXAMPLES = 2000
MAX_EXAMPLES = 0          # 0 = all examples
SEED = 42

# Trainer settings from general_trainer.py.
EPOCHS = 3.0
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = None                 # None -> 2e-4 for LoRA, 2e-5 for full fine-tune
MAX_STEPS = -1            # -1 = full run
EVAL_STEPS = 1000
KEEP_BEST = False
OVERWRITE_OUTPUT_DIR = True

# LoRA is the memory-safe default. Set NO_LORA=True for a full fine-tune.
NO_LORA = False
LORA_R = 64
LORA_ALPHA = None         # None -> 2 * LORA_R
OPTIM = "auto"            # auto -> adamw_torch for LoRA, memory-light optimizer for full fine-tune
GRAD_CHECKPOINTING = "auto"  # "auto" | "on" | "off"

## Parser helpers

This is the notebook version of `parser.py`.

In [ ]:
HEADER_RE = re.compile(r"^\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2})\] (.+)$")
TS_FORMAT = "%Y-%m-%d %H:%M"
BANNER_RE = re.compile(r"^(={5,}|Guild:|Channel:)")
MARKER_RE = re.compile(r"^\{[A-Za-z ]+\}$")
SYSTEM_MESSAGES = {"Joined the server.", "Pinned a message."}
URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"<@!?\d+>|<@&\d+>|<#\d+>")
CUSTOM_EMOJI_RE = re.compile(r"<a?(:[A-Za-z0-9_]+:)\d+>")
MULTI_BLANK_RE = re.compile(r"\n\s*\n+")


def clean_content(lines):
    """Clean raw message lines by dropping media/embed blocks and Discord markup."""
    kept = []
    in_block = False
    for line in lines:
        stripped = line.strip()
        if MARKER_RE.match(stripped):
            in_block = True
            continue
        if in_block:
            if stripped == "":
                in_block = False
            continue
        kept.append(line)

    text = "\n".join(kept)
    text = URL_RE.sub("", text)
    text = MENTION_RE.sub("", text)
    text = CUSTOM_EMOJI_RE.sub(r"\1", text)
    text = "\n".join(line.strip() for line in text.split("\n"))
    text = MULTI_BLANK_RE.sub("\n", text)
    return text.strip()


def parse_messages(path, drop_authors, min_chars):
    """Yield (datetime, author, content) for every real message in the export."""
    raw = Path(path).read_text(encoding="utf-8-sig")
    lines = raw.splitlines()

    cur_ts = None
    cur_author = None
    buf = []
    raw_count = 0

    def flush():
        if cur_author is None:
            return None
        content = clean_content(buf)
        if not content or len(content) < min_chars:
            return None
        if content in SYSTEM_MESSAGES:
            return None
        if cur_author in drop_authors:
            return None
        try:
            dt = datetime.strptime(cur_ts, TS_FORMAT)
        except (TypeError, ValueError):
            return None
        return dt, cur_author, content

    for line in lines:
        match = HEADER_RE.match(line)
        if match:
            raw_count += 1
            out = flush()
            if out is not None:
                yield out
            cur_ts, cur_author = match.group(1), match.group(2).strip()
            buf = []
        elif cur_author is not None:
            if BANNER_RE.match(line):
                continue
            buf.append(line)

    out = flush()
    if out is not None:
        yield out
    parse_messages.raw_count = raw_count


def build_sessions(messages, gap_minutes, merge_minutes, max_turns):
    """Group merged turns into sessions split on conversational time gaps."""
    gap = timedelta(minutes=gap_minutes)
    merge = timedelta(minutes=merge_minutes)
    sessions = []
    cur = []
    prev_dt = None

    for dt, author, content in messages:
        new_session = prev_dt is None or (dt - prev_dt) > gap or len(cur) >= max_turns
        if new_session and cur:
            if len(cur) >= 2:
                sessions.append(cur)
            cur = []

        if cur and cur[-1]["author"] == author and prev_dt is not None and (dt - prev_dt) <= merge:
            cur[-1]["content"] += "\n" + content
        else:
            cur.append({"author": author, "content": content})
        prev_dt = dt

    if len(cur) >= 2:
        sessions.append(cur)
    return sessions


def write_conversations_jsonl(input_path, output_path, gap_minutes, merge_minutes, max_turns, min_chars, drop_authors):
    """Parse the raw export and write one JSON conversation session per line."""
    drop_authors = set(drop_authors)
    messages = list(parse_messages(input_path, drop_authors, min_chars))
    sessions = build_sessions(messages, gap_minutes, merge_minutes, max_turns)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    n_turns = 0
    authors = set()
    with output_path.open("w", encoding="utf-8") as handle:
        for session_id, turns in enumerate(sessions):
            n_turns += len(turns)
            authors.update(turn["author"] for turn in turns)
            payload = {"session_id": session_id, "turns": turns}
            handle.write(json.dumps(payload, ensure_ascii=False) + "\n")

    return {
        "raw_message_headers": getattr(parse_messages, "raw_count", 0),
        "kept_messages": len(messages),
        "sessions_written": len(sessions),
        "turns_after_merge": n_turns,
        "unique_authors": len(authors),
        "avg_turns_per_session": (n_turns / len(sessions)) if sessions else 0.0,
        "output": str(output_path),
    }

## Prepare conversations

This cell reuses `data/conversations.jsonl` when it exists. Set `FORCE_REPARSE = True` in the config cell to rebuild it from `dataset.txt`.

In [ ]:
conversations_path = Path(CONVERSATIONS_OUTPUT)
raw_input_path = Path(RAW_INPUT)

if FORCE_REPARSE or not conversations_path.exists():
    if not raw_input_path.exists():
        raise FileNotFoundError(
            f"No parsed dataset at {conversations_path} and no raw export at {raw_input_path}. "
            "Put your Discord export at RAW_INPUT or point CONVERSATIONS_OUTPUT at an existing jsonl file."
        )

    stats = write_conversations_jsonl(
        input_path=raw_input_path,
        output_path=conversations_path,
        gap_minutes=GAP_MINUTES,
        merge_minutes=MERGE_MINUTES,
        max_turns=MAX_TURNS,
        min_chars=MIN_CHARS,
        drop_authors=DROP_AUTHORS,
    )
    for key, value in stats.items():
        print(f"{key:20}: {value}")
else:
    print(f"using existing parsed conversations: {conversations_path}")

## Training data helpers

This is the notebook version of the dataset-building code from `general_trainer.py`.

In [ ]:
def load_sessions(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line)["turns"] for line in handle if line.strip()]


def build_examples(sessions, context_turns):
    """Yield (context_turns_list, target_content) sliding over each session."""
    for turns in sessions:
        for i in range(1, len(turns)):
            context = turns[max(0, i - context_turns):i]
            target = turns[i]["content"]
            if target:
                yield context, target


def make_tokenize_fn(tokenizer, max_len):
    """Tokenize one example and mask every token except the target reply."""
    eos = tokenizer.eos_token or ""

    def tokenize(context, target):
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": render_transcript(context)},
        ]
        prompt_text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompt_ids = tokenizer(prompt_text, add_special_tokens=True)["input_ids"]
        full_ids = tokenizer(prompt_text + target + eos, add_special_tokens=True)["input_ids"]

        labels = list(full_ids)
        for idx in range(min(len(prompt_ids), len(full_ids))):
            labels[idx] = -100

        if len(full_ids) > max_len:
            full_ids = full_ids[-max_len:]
            labels = labels[-max_len:]

        if all(token_id == -100 for token_id in labels):
            return None

        return {
            "input_ids": full_ids,
            "attention_mask": [1] * len(full_ids),
            "labels": labels,
        }

    return tokenize


def build_dataset(sessions, tokenizer, context_turns, max_len, max_examples, label="train"):
    tokenize = make_tokenize_fn(tokenizer, max_len)
    rows = []

    for n, (context, target) in enumerate(build_examples(sessions, context_turns), 1):
        row = tokenize(context, target)
        if row is not None:
            rows.append(row)

        if n % 20000 == 0:
            print(f"tokenizing {label}... {len(rows)} examples", flush=True)

        if max_examples and len(rows) >= max_examples:
            break

    return Dataset.from_list(rows)

## Derived training settings

In [ ]:
if GRAD_CHECKPOINTING not in {"auto", "on", "off"}:
    raise ValueError("GRAD_CHECKPOINTING must be 'auto', 'on', or 'off'")

use_lora = not NO_LORA
lr = LR if LR is not None else (2e-4 if use_lora else 2e-5)
lora_alpha = LORA_ALPHA if LORA_ALPHA is not None else 2 * LORA_R

bf16_supported = bool(torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)())
use_bf16 = bf16_supported
use_fp16 = torch.cuda.is_available() and not use_bf16
dtype = torch.bfloat16 if use_bf16 else (torch.float16 if use_fp16 else torch.float32)
grad_ckpt = {"auto": not use_lora, "on": True, "off": False}[GRAD_CHECKPOINTING]

if OPTIM != "auto":
    optim = OPTIM
elif use_lora:
    optim = "adamw_torch"
elif importlib.util.find_spec("bitsandbytes") is not None:
    optim = "paged_adamw_8bit"
else:
    optim = "adafactor"

print(f"use_lora={use_lora}")
print(f"lr={lr}")
print(f"optim={optim}")
print(f"dtype={dtype}")
print(f"grad_ckpt={grad_ckpt}")

## Tokenizer and datasets

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Truncation is handled manually in make_tokenize_fn.
tokenizer.model_max_length = int(1e9)

sessions = load_sessions(CONVERSATIONS_OUTPUT)
if not sessions:
    raise ValueError(f"No sessions found in {CONVERSATIONS_OUTPUT}")

random.Random(SEED).shuffle(sessions)
n_val = int(len(sessions) * VAL_FRAC)
val_sessions = sessions[:n_val]
train_sessions = sessions[n_val:]

if not train_sessions:
    raise ValueError("Training split is empty. Lower VAL_FRAC or add more sessions.")

print("Building dataset...")
train_ds = build_dataset(train_sessions, tokenizer, CONTEXT_TURNS, MAX_LEN, MAX_EXAMPLES)
val_ds = None
if val_sessions:
    val_ds = build_dataset(val_sessions, tokenizer, CONTEXT_TURNS, MAX_LEN, MAX_VAL_EXAMPLES, label="val")

if len(train_ds) == 0:
    raise ValueError("Training dataset is empty after tokenization.")

print(f"sessions total      : {len(sessions)}")
print(f"training sessions  : {len(train_sessions)}")
print(f"validation sessions: {len(val_sessions)}")
print(f"training examples  : {len(train_ds)}")
print(f"validation examples: {len(val_ds) if val_ds is not None else 0}")

## Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)
model.config.pad_token_id = tokenizer.pad_token_id

if grad_ckpt:
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

if use_lora:
    from peft import LoraConfig, get_peft_model

    lora = LoraConfig(
        r=LORA_R,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )
    model = get_peft_model(model, lora)
    if grad_ckpt:
        model.enable_input_require_grads()
    model.print_trainable_parameters()

collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

## Trainer

In [ ]:
do_eval = val_ds is not None and len(val_ds) > 0
valid_training_args = set(inspect.signature(TrainingArguments.__init__).parameters)

ta_kwargs = dict(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=OVERWRITE_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=lr,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0 if use_lora else 0.01,
    optim=optim,
    gradient_checkpointing=grad_ckpt,
    gradient_checkpointing_kwargs={"use_reentrant": False} if grad_ckpt else None,
    bf16=use_bf16,
    fp16=use_fp16,
    logging_steps=50,
    eval_steps=EVAL_STEPS if do_eval else None,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=3,
    load_best_model_at_end=do_eval and KEEP_BEST,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=[],
    seed=SEED,
    data_seed=SEED,
)

# Transformers renamed evaluation_strategy to eval_strategy. Pick whichever this install supports.
strategy_value = "steps" if do_eval else "no"
if "eval_strategy" in valid_training_args:
    ta_kwargs["eval_strategy"] = strategy_value
elif "evaluation_strategy" in valid_training_args:
    ta_kwargs["evaluation_strategy"] = strategy_value
elif do_eval:
    print("[warn] installed transformers does not expose an evaluation strategy argument; eval is disabled")

ta_kwargs = {key: value for key, value in ta_kwargs.items() if value is not None}
unsupported = [key for key in ta_kwargs if key not in valid_training_args]
if unsupported:
    print(f"[info] transformers ignoring unsupported args: {unsupported}")

training_args = TrainingArguments(
    **{key: value for key, value in ta_kwargs.items() if key in valid_training_args}
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=val_ds if do_eval else None,
)

print(training_args)

## Train and save

In [ ]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"saved -> {OUTPUT_DIR}")
train_result